# N130 · 线性递推与矩阵快速幂

**目标：** 把固定维状态递推写成线性变换。

**先修：** N098, N128, N082。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 状态向量；转移矩阵；矩阵乘法；单位矩阵；模数；维度成本。

**配套讲解来源：** [同名逐章意见](../../comment/130_matrix_exponentiation_recurrences.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章要解决的问题是：**像斐波那契那种"下一项由前几项线性组合而来"的递推数列，当 n 大到 10¹⁸ 时，一步步递推（O(n) 轮）根本跑不完，怎么把轮数压到"约 log n 轮"？** 答案是把递推写成矩阵乘法，再用上一章（N128）的快速幂思想对矩阵求幂。

给你具体到数字的例子：

- 输入：`n = 10`
- 输出：`fib_matrix(10) = 55`
- 为什么是 55？斐波那契数列 0,1,1,2,3,5,8,13,21,34,55，第 10 项是 55。但 `fib_matrix` 不是循环加 10 次得到的，而是把 10 写成二进制 1010，只做了 4 轮"矩阵平方"就拿到了 A¹⁰，再从矩阵里读出 55。n = 10¹⁸ 时也只是约 60 轮，规模无关紧要。

第二个例子是元音串计数：长度为 n 的字符串，每个字符是 a/e/i/o/u，但相邻字母要遵守"谁后面能跟谁"的规则（a 后面只能是 e，等等），数一共有多少种。输入 `n = 5`，输出 `count_vowel_permutation(5) = 68`。这题的状态转移同样是线性的，于是同一套矩阵快速幂直接复用，配上 mod 10⁹+7。

## 2. 基础知识：先读懂这些词

- **线性递推**：每一项等于前面若干项的固定加权和不加常数。斐波那契 F(n)=F(n−1)+F(n−2) 是最简单的例子。它好就好在能写成矩阵形式。
- **状态向量**：把递推需要的"最近几个值"打包成一列。比如斐波那契要最近两项，状态向量就是 x[t] = (F(t+1), F(t))。
- **转移矩阵**：把"从 x[t] 变到 x[t+1]"的固定规则写成的矩阵 A，满足 x[t+1] = A·x[t]。有了它，x[t] = Aᵗ·x[0]——递推 n 步就变成矩阵 A 的 n 次幂。
- **矩阵乘法**：两个矩阵相乘，结果的第 i 行第 j 列等于左矩阵第 i 行与右矩阵第 j 列对应相乘再求和。它恰好表达了"新状态的每个分量 = 各旧状态分量的加权和"。
- **单位矩阵（identity）**：对角线是 1、其余是 0 的矩阵，作用相当于数字里的 1——任何矩阵乘它都保持不变。矩阵的零次幂必须是它（A⁰ = I），不是零矩阵。
- **结合律与不交换**：矩阵乘法满足结合律 (AB)C = A(BC)，所以快速幂的"反复平方"对矩阵照样成立；但一般 AB ≠ BA，所以乘法次序不能随意交换——快速幂代码里始终保持"结果在左、底数在右"的次序。
- **模数（mod）**：计数题的答案巨大，算的过程中随时 % (10⁹+7) 压住数字。本章 `matmul_mod` 允许 `mod=None` 表示"不取模、要精确大整数"（fib 用这条路）。
- **维度成本（d³ 因子）**：d×d 矩阵相乘一次要做约 d³ 次乘加。矩阵快速幂省的是"步数"（n → log n），但每次的价钱随状态维度立方增长，所以只有状态维度小（比如 2 或 5）时才划算。

## 3. 思路推导：从小例子开始

- **Step 1：把递推关系打包成"状态向量 + 转移矩阵"。** 斐波那契要最近两项才知道下一项，于是取 x[t] = (F(t+1), F(t))，可以验证 x[t+1] = A·x[t]，其中 A = [[1,1],[1,0]]：新向量的第一个分量 = 1·F(t+1) + 1·F(t) = F(t+2)，第二个分量 = 1·F(t+1) + 0·F(t) = F(t+1)，正是新状态的正确内容。
- **Step 2：多步递推 = 矩阵的幂。** x[n] = Aⁿ·x[0]，一步一顿的递推被一次性压缩。
- **Step 3：对矩阵用二进制快速幂。** 和 N128 的数字快速幂同构：指数 n 每轮右移一位，底数矩阵不断自平方；当前指数位是 1 就把底数矩阵乘进结果。合法性来自结合律。
- **Step 4：从幂矩阵里读答案。** 斐波那契的答案 F(n) 出现在 Aⁿ 的 [0][1] 位置（原因见下面表格演示）。
- **Step 5：计数题照搬这套流程。** 元音题的"状态"是最后一个字母是哪个元音（5 维），转移规则"谁能跟谁"写成 5×5 的 0/1 矩阵，答案 = 矩阵 n−1 次幂的所有元素之和（再取模）。

**手算演示一：A 的幂和 F(n)**（“运行示例”部分 的表格正是这些数据，矩阵 [[1,1],[1,0]]，n 从 0 到 5）：

| 指数 n | 矩阵幂 Aⁿ | F(n) |
|--------|-----------|------|
| 0 | [[1,0],[0,1]] | 0 |
| 1 | [[1,1],[1,0]] | 1 |
| 2 | [[2,1],[1,1]] | 1 |
| 3 | [[3,2],[2,1]] | 2 |
| 4 | [[5,3],[3,2]] | 3 |
| 5 | [[8,5],[5,3]] | 5 |

盯着 [0][1] 那个位置看：1、1、2、3、5，正是斐波那契！这不是巧合：Aⁿ·(1,0) 相当于取 Aⁿ 的第一列，而 x[n−1] = Aⁿ⁻¹·(1,0) = (F(n), F(n−1))，所以 Aⁿ 的 [0][1] 元素一直是 F(n)。`fib_matrix` 里 `[0][1]` 读的就是这个。

**手算演示二：元音计数从 n=1 推到 n=5。** 状态向量记录"以每个元音结尾的串数"，初始（n=1）每种一个：(1,1,1,1,1)，总和 5。规则是 a 后可跟 e；e 后可跟 a 或 i；i 后可跟 a/e/o/u；o 后可跟 i 或 u；u 后可跟 a。每轮按规则搬运：

| 长度 n | (a, e, i, o, u) 结尾计数 | 总和 |
|--------|--------------------------|------|
| 1 | (1, 1, 1, 1, 1) | 5 |
| 2 | (3, 2, 2, 1, 2) | 10 |
| 3 | (6, 5, 3, 2, 3) | 19 |
| 4 | (11, 9, 7, 3, 5) | 35 |
| 5 | (21, 18, 12, 7, 10) | 68 |

以 n=2 那行为例核对：能以 a 结尾的旧串是以 e、i、u 结尾的三种来源 1+1+1=3；只能以 o 结尾的旧来源 i 一种，得 1。一轮"搬运"就是一次矩阵乘向量；`count_vowel_permutation` 把 n−1 次搬运合成一次矩阵 n−1 次幂，最后 `sum(map(sum,power))` 把"各种开头 × 各种结尾"的总数一次性加出来。

## 4. 核心代码：matmul_mod

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def matmul_mod(a, b, mod):
    if mod is not None and mod <= 0:
        raise ValueError('positive modulus required')
    if not a or not b:
        return []
    rows, inner, cols = (len(a), len(a[0]), len(b[0]))
    if any((len(row) != inner for row in a)) or len(b) != inner or any((len(row) != cols for row in b)):
        raise ValueError('incompatible rectangular matrices')
    out = [[0] * cols for _ in range(rows)]
    for i in range(rows):
        for k in range(inner):
            for j in range(cols):
                out[i][j] += a[i][k] * b[k][j]
                if mod is not None:
                    out[i][j] %= mod
    return out
```

### 逐段读懂代码

“分段实现”部分 有四个函数。

**函数一：`matmul_mod(a, b, mod)` —— 带模的矩阵乘法。**

```python
def matmul_mod(a, b, mod):
    if mod is not None and mod <= 0:
        raise ValueError('positive modulus required')
    if not a or not b:
        return []
    rows, inner, cols = (len(a), len(a[0]), len(b[0]))
    if any((len(row) != inner for row in a)) or len(b) != inner or any((len(row) != cols for row in b)):
        raise ValueError('incompatible rectangular matrices')
    out = [[0] * cols for _ in range(rows)]
    for i in range(rows):
        for k in range(inner):
            for j in range(cols):
                out[i][j] += a[i][k] * b[k][j]
                if mod is not None:
                    out[i][j] %= mod
    return out
```

- `if mod is not None and mod<=0: raise ValueError(...)`：mod 是可选参数，传 None 表示"不取模求精确整数"；但只要传了就必须是正数，mod=0 会让取模运算除零。
- `if not a or not b: return []`：空矩阵相乘按约定返回空，避免后面 `len(a[0])` 越界。
- 三行形状检查：左矩阵必须是 rows×inner 的矩形（每行同长），右矩阵必须行数等于 inner 且每行同长（inner×cols），否则数学上乘不起来，报错拒绝。`any(len(row)!=inner for row in a)` 就是在数"有没有哪行长度不齐"。
- 三重循环 `i, k, j`：i 扫结果的行，j 扫结果的列，k 是求和下标。这里刻意把 k 放在中层——按"左矩阵一行 × 右矩阵一列"的数学定义等价展开成逐项累加 `out[i][j]+=a[i][k]*b[k][j]`，同时每加一次就 `%mod`，中间数字永远不膨胀。
- 注意循环变量顺序 i→k→j 而不是 i→j→k：两者算出的结果一样，但 i-k-j 顺序对缓存访问更友好（连续读 b 的同一行），这是写法上的小讲究。

**函数二：`matpow_mod(a, n, mod)` —— 矩阵快速幂。**

```python
def matpow_mod(a, n, mod):
    size = len(a)
    if n < 0 or any((len(row) != size for row in a)):
        raise ValueError('square matrix and nonnegative exponent required')
    result = [[int(i == j) if mod is None else int(i == j) % mod for j in range(size)] for i in range(size)]
    base = [row[:] for row in a]
    while n:
        if n & 1:
            result = matmul_mod(result, base, mod)
        base = matmul_mod(base, base, mod)
        n >>= 1
    return result
```

- `if n<0 or any(len(row)!=size for row in a): raise ValueError(...)`：一次校验两件事——指数非负（负指数要矩阵求逆，超纲）且输入是方阵（每行长度都等于行数）。只有方阵谈幂才有意义。
- `result=[[int(i==j) ...]]`：把 result 初始化成单位矩阵。`int(i==j)` 是紧凑写法：i==j 为 True 转 1，否则 0，正好铺出对角线为 1 的单位阵；`%mod` 分支伺候 mod=1 之类的退化情形（单位阵也取模）。
- `base=[row[:] for row in a]`：把输入矩阵逐行浅拷贝一份。不直接引用 a 是防止后续平方覆盖调用方手里的原矩阵——列表是可变对象，这种防御性拷贝是好习惯。
- `while n:` + `if n&1: result=matmul_mod(result,base,mod)` + `base=matmul_mod(base,base,mod); n>>=1`：和 N128 数字快速幂逐行对应的四件套——查最低位、该乘就乘（注意次序固定是 result 在左 base 在右，因为矩阵不交换）、底数自平方、指数右移。

**函数三：`fib_matrix(n)` —— 斐波那契第 n 项。**

```python
def fib_matrix(n):
    if n < 0:
        raise ValueError('nonnegative index required')
    return matpow_mod([[1, 1], [1, 0]], n, None)[0][1]
```

- `if n<0: raise ValueError(...)`：挡住负下标，序列从 0 号开始。
- 一行三件事：写出转移矩阵 A；求 A 的 n 次幂（mod 传 None 要精确整数）；取结果的 [0][1] 位置。为什么是 [0][1]？第三节表格已经演示，那个位置永远是 F(n)。

**函数四：`count_vowel_permutation(n)` —— 元音排列计数（mod 10⁹+7）。**

```python
def count_vowel_permutation(n):
    if n < 0:
        raise ValueError('nonnegative length required')
    if n == 0:
        return 1
    edges = {0: [1], 1: [0, 2], 2: [0, 1, 3, 4], 4: [0], 3: [2, 4]}
    matrix = [[0] * 5 for _ in range(5)]
    for old, nexts in edges.items():
        for new in nexts:
            matrix[new][old] = 1
    power = matpow_mod(matrix, n - 1, 10 ** 9 + 7)
    return sum(map(sum, power)) % (10 ** 9 + 7)
```

- `if n<0: raise ValueError(...)` 和 `if n==0: return 1`：前者挡负长度；后者是边界约定——空串有且只有 1 个（空串本身），直接给答案不走矩阵（n−1 会变成负指数被 matpow 拒绝）。
- `edges={0:[1],1:[0,2],2:[0,1,3,4],3:[2,4],4:[0]}`：用编号 0..4 表示 a,e,i,o,u，字典记录"old 后面可以接哪些 new"。比如键 2（i）后面可以接 0,1,3,4（a,e,o,u）。
- `matrix[new][old]=1`：把边表翻译成 5×5 转移矩阵，**行是新状态、列是旧状态**（“正确性与复杂度”部分 特意强调这个约定）。置 1 表示"从 old 一步走到 new 的走法有一种"。
- `power=matpow_mod(matrix,n-1,10**9+7)`：长度 n 的串有 n−1 次"相邻转移"，即矩阵的 n−1 次幂；power[i][j] 的含义是"从字母 j 出发走 n−1 步、以字母 i 结尾的串数"。
- `return sum(map(sum,power))%(10**9+7)`：总串数要对所有"开头字母 × 结尾字母"组合求和，而矩阵每个元素本身就是模过的，把全部 25 个元素加起来再取一次模即可。`map(sum,power)` 先把每行求和，外层 sum 再合计。n=1 时幂是单位矩阵，元素和恰为 5，与"5 个单字母串"一致。

## 5. 分段实现：按顺序运行

**本章接口：** `matmul_mod(a, b, mod)`、`matpow_mod(a, n, mod)`、`fib_matrix(n)`、`count_vowel_permutation(n)`

### matmul_mod

In [1]:
def matmul_mod(a, b, mod):
    if mod is not None and mod <= 0:
        raise ValueError('positive modulus required')
    if not a or not b:
        return []
    rows, inner, cols = (len(a), len(a[0]), len(b[0]))
    if any((len(row) != inner for row in a)) or len(b) != inner or any((len(row) != cols for row in b)):
        raise ValueError('incompatible rectangular matrices')
    out = [[0] * cols for _ in range(rows)]
    for i in range(rows):
        for k in range(inner):
            for j in range(cols):
                out[i][j] += a[i][k] * b[k][j]
                if mod is not None:
                    out[i][j] %= mod
    return out

### matpow_mod

In [2]:
def matpow_mod(a, n, mod):
    size = len(a)
    if n < 0 or any((len(row) != size for row in a)):
        raise ValueError('square matrix and nonnegative exponent required')
    result = [[int(i == j) if mod is None else int(i == j) % mod for j in range(size)] for i in range(size)]
    base = [row[:] for row in a]
    while n:
        if n & 1:
            result = matmul_mod(result, base, mod)
        base = matmul_mod(base, base, mod)
        n >>= 1
    return result

### fib_matrix

In [3]:
def fib_matrix(n):
    if n < 0:
        raise ValueError('nonnegative index required')
    return matpow_mod([[1, 1], [1, 0]], n, None)[0][1]

### count_vowel_permutation

In [4]:
def count_vowel_permutation(n):
    if n < 0:
        raise ValueError('nonnegative length required')
    if n == 0:
        return 1
    edges = {0: [1], 1: [0, 2], 2: [0, 1, 3, 4], 3: [2, 4], 4: [0]}
    matrix = [[0] * 5 for _ in range(5)]
    for old, nexts in edges.items():
        for new in nexts:
            matrix[new][old] = 1
    power = matpow_mod(matrix, n - 1, 10 ** 9 + 7)
    return sum(map(sum, power)) % (10 ** 9 + 7)

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [5]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

matrix=[[1,1],[1,0]]
show_table(['指数','矩阵幂','F(n)'],[(n,matpow_mod(matrix,n,None),fib_matrix(n)) for n in range(6)])

指数,矩阵幂,F(n)
0,"[[1, 0], [0, 1]]",0
1,"[[1, 1], [1, 0]]",1
2,"[[2, 1], [1, 1]]",1
3,"[[3, 2], [2, 1]]",2
4,"[[5, 3], [3, 2]]",3
5,"[[8, 5], [5, 3]]",5


## 7. 正确性、适用条件与复杂度

转移矩阵的列代表旧状态、行代表新状态。乘法展开正好累加所有一步来源，矩阵幂相乘拼接独立转移步。快速幂保持已累积结果与尚未处理的幂因子乘积不变；零次幂必须为单位阵而非零阵。

**代价：** d维稠密矩阵幂O(d³ log n)次算术操作、O(d²)空间。fib_matrix用mod=None得到精确整数，结果位数为Θ(n)，因此不能称总位复杂度O(log n)。元音计数采用模10^9+7。

### 展开理解

**矩阵一行一列的乘出来的是什么？** 转移矩阵列表示旧状态、行表示新状态时，乘法展开式的每一项都是"某个旧分量 × 一步能不能这么走"，求和恰好把"能到达新状态的所有一步来源"全加了一遍，不漏也不多。矩阵幂则是把一步转移复用拼接：Aᵐ·Aⁿ = A^(m+n)，连续乘 n 个 A 就等于走 n 步。

**快速幂循环里什么保持不变？** 任意时刻，"已经乘进 result 的部分"乘上"base 里还没消费的 A 的幂"始终等于 A 的原始 n 次幂。指数每轮折半，这个乘积纹丝不动，所以 n 归零时 result 独自等于 Aⁿ。

**为什么零次幂必须是单位矩阵？** A⁰ 的含义是"走零步"，任何向量走零步都该保持原样，而"乘了等于没乘"的矩阵只有单位矩阵。如果误初始化成零矩阵，所有结果会全盘归零。测试里 `matpow_mod(...,0,17)==[[1,0],[0,1]]` 查的就是这个。

**复杂度（结合具体数字）：** d×d 矩阵乘一次约 d³ 次乘加，快速幂做约 log₂n 轮，总计 O(d³ log n)。斐波那契 d=2，一次矩阵乘 8 次乘加，n = 10¹⁸ 时约 60 轮，总共几百次操作，眨眼完成。元音题 d=5，一次 125 次乘加，同样轻松。但 “正确性与复杂度”部分 的警告要认真听：**如果状态维度 d 很大**（比如 d=200），d³ = 8×10⁶，单次矩阵乘就比一轮普通 DP 还贵，矩阵快速幂反而输给线性递推。另一个坑是 fib 走 `mod=None` 精确整数路线：F(n) 的位数和 n 成正比（F(10⁵) 约两万位），大整数乘法按位数收费，所以总代价不能只看"log n 轮"，这就是 “正确性与复杂度”部分 说"不能称总位复杂度 O(log n)"的意思。

## 8. 单元测试：每个用例在检查什么

1. `assert matpow_mod([[2,3],[4,5]],0,17)==[[1,0],[0,1]]` —— **边界值**：零次幂必须返回单位矩阵（哪怕 mod=17 也要是模过后的单位阵），这检验 result 的初始化写法。
2. `assert fib_matrix(0)==0 and fib_matrix(10)==55` —— **正常值 + 边界值**：F(0)=0 是序列头（此时 A⁰ 是单位阵，[0][1] 位置为 0，恰好读出 0），F(10)=55 抽查中段。F(1)=1 这类值不用单独抽查，下一条循环会覆盖到。
3. `count_vowel_permutation` 三个抽查：n=1 → 5（五个元音各一个）、n=2 → 10（第三节表格验证过）、n=5 → 68（表格最后一行的总和）。
4. 斐波那契长跑对账：`x,y=0,1` 起步，循环 120 次，每次断言 `fib_matrix(n)==x` 然后 `x,y=y,x+y` 前滚——拿最朴素的迭代法当裁判，n 从 0 一路对到 119，顺带验证了精确整数路线（无 mod）在大到 F(119)≈3×10²⁴ 时也不出错。
5. 元音题长跑对账：用 dp 数组手工模拟转移（`for old,targets in edges.items(): for new in targets: nxt[new]+=dp[old]`），n 从 1 对到 39，矩阵路线和线性 DP 路线必须逐步一致——这同时检验了矩阵的行列方向有没有摆反（摆反的话多数 n 会立刻露馅）。
6. `assert matmul_mod([[1,2,3]],[[4],[5],[6]],97)==[[32]]` —— **形状抽查**：1×3 乘 3×1 得 1×1，值 1×4+2×5+3×6 = 4+10+18 = 32，顺带确认矩形（非方阵）乘法也走通。

In [6]:
assert matpow_mod([[2, 3], [4, 5]], 0, 17) == [[1, 0], [0, 1]]

assert fib_matrix(0) == 0 and fib_matrix(10) == 55

assert count_vowel_permutation(1) == 5 and count_vowel_permutation(2) == 10 and (count_vowel_permutation(5) == 68)

x, y = (0, 1)

for n in range(120):
    assert fib_matrix(n) == x
    x, y = (y, x + y)

edges = {0: [1], 1: [0, 2], 2: [0, 1, 3, 4], 3: [2, 4], 4: [0]}

dp = [1] * 5

for n in range(1, 40):
    assert count_vowel_permutation(n) == sum(dp) % (10 ** 9 + 7)
    nxt = [0] * 5
    for old, targets in edges.items():
        for new in targets:
            nxt[new] += dp[old]
    dp = nxt

assert matmul_mod([[1, 2, 3]], [[4], [5], [6]], 97) == [[32]]

print('N130: 所有本章断言通过')

N130: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 从三阶递推构造矩阵。

**练习 2：** 说明复杂度含矩阵维度而不只log n。

<details>
<summary>展开思路提示（不是完整答案）</summary>

**练习 1（三阶递推构造矩阵）：** 提示——像"Tribonacci"T(n) = T(n−1)+T(n−2)+T(n−3) 这种，状态向量取"最近三项" (T(n+2), T(n+1), T(n))，转移矩阵是 3×3：第一行三个 1（新头是三项之和），第二行是 1,0,0（把旧第二项挪到第二位），第三行 0,1,0。先手算矩阵乘一次向量验证挪位正确（比如从 (2,1,1) 推出下一列），再对照 `matpow_mod` 求幂和暴力递推各算 T(10) 到 T(20)。边界要想清楚 T(0),T(1),T(2) 的初值怎么对应初始状态向量。

**练习 2（复杂度里除了 log n 还有维度）：** 提示——做一次对比实验：固定递推是斐波那契（d=2）和元音题（d=5），分别记下 n = 10⁶ 时矩阵快速幂和线性 DP 的耗时感觉；再构造一个假想的 d=100 状态（比如 100 项线性组合），算一笔账：矩阵乘一次 10⁶ 次操作 × log n ≈ 20 轮 = 2×10⁷，而线性 DP 每步 100 次加法 × 10⁶ 步 = 10⁸——诶，这时矩阵还略赢；把 d 加到 300 再算一遍，结论就翻过来了。把这个"翻车的 d"找出来，你就理解 O(d³ log n) 对比 O(d·n) 的分界线了。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [7]:
# 完整实现：本单元格不依赖其他单元格。
def matmul_mod(a, b, mod):
    if mod is not None and mod <= 0:
        raise ValueError('positive modulus required')
    if not a or not b:
        return []
    rows, inner, cols = (len(a), len(a[0]), len(b[0]))
    if any((len(row) != inner for row in a)) or len(b) != inner or any((len(row) != cols for row in b)):
        raise ValueError('incompatible rectangular matrices')
    out = [[0] * cols for _ in range(rows)]
    for i in range(rows):
        for k in range(inner):
            for j in range(cols):
                out[i][j] += a[i][k] * b[k][j]
                if mod is not None:
                    out[i][j] %= mod
    return out

def matpow_mod(a, n, mod):
    size = len(a)
    if n < 0 or any((len(row) != size for row in a)):
        raise ValueError('square matrix and nonnegative exponent required')
    result = [[int(i == j) if mod is None else int(i == j) % mod for j in range(size)] for i in range(size)]
    base = [row[:] for row in a]
    while n:
        if n & 1:
            result = matmul_mod(result, base, mod)
        base = matmul_mod(base, base, mod)
        n >>= 1
    return result

def fib_matrix(n):
    if n < 0:
        raise ValueError('nonnegative index required')
    return matpow_mod([[1, 1], [1, 0]], n, None)[0][1]

def count_vowel_permutation(n):
    if n < 0:
        raise ValueError('nonnegative length required')
    if n == 0:
        return 1
    edges = {0: [1], 1: [0, 2], 2: [0, 1, 3, 4], 3: [2, 4], 4: [0]}
    matrix = [[0] * 5 for _ in range(5)]
    for old, nexts in edges.items():
        for new in nexts:
            matrix[new][old] = 1
    power = matpow_mod(matrix, n - 1, 10 ** 9 + 7)
    return sum(map(sum, power)) % (10 ** 9 + 7)

# 示例与回归测试
assert matpow_mod([[2, 3], [4, 5]], 0, 17) == [[1, 0], [0, 1]]

assert fib_matrix(0) == 0 and fib_matrix(10) == 55

assert count_vowel_permutation(1) == 5 and count_vowel_permutation(2) == 10 and (count_vowel_permutation(5) == 68)

x, y = (0, 1)

for n in range(120):
    assert fib_matrix(n) == x
    x, y = (y, x + y)

edges = {0: [1], 1: [0, 2], 2: [0, 1, 3, 4], 3: [2, 4], 4: [0]}

dp = [1] * 5

for n in range(1, 40):
    assert count_vowel_permutation(n) == sum(dp) % (10 ** 9 + 7)
    nxt = [0] * 5
    for old, targets in edges.items():
        for new in targets:
            nxt[new] += dp[old]
    dp = nxt

assert matmul_mod([[1, 2, 3]], [[4], [5], [6]], 97) == [[32]]

print('N130: 所有本章断言通过')

N130: 所有本章断言通过


## 11. 代表题与迁移

- **509. Fibonacci Number（斐波那契数）**：直接问 F(n)——本章 `fib_matrix` 的原型，小 n 用循环就够，大 n 才显出矩阵快速幂的威力（extension，作为矩阵法的迁移练习）。
- **1137. N-th Tribonacci Number（第 N 个泰波那契数）**：三项求和递推——练"把三阶递推的转移矩阵亲手搭出来"，正是练习 1 的题目化。
- **1220. Count Vowels Permutation（统计元音字母序列的数目）**：相邻字母带规则的计数——`count_vowel_permutation` 的原型题（canonical），练"邻接规则 → 转移矩阵 → 幂 → 求和取模"的全流程。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。